---
title: Data Preparation
date: 2026-09-08
---


# Data Preparation

Tahap **Data Preparation** dalam kerangka kerja **CRISP-DM** mengolah data harian kualitas udara untuk wilayah **Bangkalan, Jawa Timur**. Dataset bersumber dari `data_polutan_bangkalan.csv` dan memuat **365 observasi harian**, dari **30 Agustus 2025 sampai 29 Agustus 2026**. Data mencakup parameter **CO, NO2, SO2, dan CH4**, serta kolom rerata bergerak 30 hari. Batas Area of Interest (AOI) dibaca dari `coordinates_bangkalan.json`.

Audit awal menemukan nilai kosong pada CO (**264; 72,33%**), NO2 (**254; 69,59%**), SO2 (**177; 48,49%**), dan CH4 (**364; 99,73%**). Nilai kosong diimputasi menggunakan interpolasi linear serta *backfill/forward fill* pada batas deret. Karena CH4 hanya memiliki satu nilai harian teramati, deret hasil imputasinya konstan dan perlu ditafsirkan dengan hati-hati.

### Alur Kerja Pemrosesan Data Preparation
1. **Pemuatan dan peninjauan dataset CSV** serta koordinat AOI Bangkalan.
2. **Imputasi missing values** pada empat parameter polutan.
3. **Identifikasi dan penanganan outlier** menggunakan metode **IQR (Interquartile Range)** dan **Winsorization Capping**.
4. **Ekstraksi fitur deret waktu TSFEL** untuk parameter polutan.
5. **Analisis kemiripan sinyal** menggunakan Korelasi Pearson dan Cosine Similarity.
6. **Penyimpanan dataset dan fitur** sebagai file CSV di folder `data`.


### 1. Pemuatan dan Peninjauan Awal Dataset Mentah

Dataset harian dimuat dari `data_polutan_bangkalan.csv`. Kolom tanggal dan konsentrasi harian (`CO`, `NO2`, `SO2`, dan `CH4`) digunakan sebagai variabel pengamatan; kolom rerata bergerak 30 hari tidak digunakan sebagai pengganti observasi harian. Batas AOI Bangkalan dimuat dari `coordinates_bangkalan.json`: longitude **112,7475858°E–112,7537370°E** dan latitude **7,0275990°S–7,0323882°S**.


In [ ]:
import json
import os
import pandas as pd


def get_project_root():
    curr = os.path.abspath(os.getcwd())
    while curr != os.path.dirname(curr):
        if os.path.exists(os.path.join(curr, "myst.yml")) or os.path.exists(
            os.path.join(curr, ".git")
        ):
            return curr
        curr = os.path.dirname(curr)
    return os.path.abspath(os.getcwd())


PROJECT_ROOT = get_project_root()
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
file_path = os.path.join(DATA_DIR, "data_polutan_bangkalan.csv")
coordinates_path = os.path.join(DATA_DIR, "coordinates_bangkalan.json")
df_source = pd.read_csv(file_path)
with open(coordinates_path, "r", encoding="utf-8") as file:
    aoi_geojson = json.load(file)

print(f"Dataset: {file_path}")
print(f"Ukuran dataset: {df_source.shape[0]} baris x {df_source.shape[1]} kolom")
print("Kolom:", df_source.columns.tolist())
print("\nPratinjau 8 baris pertama:")
print(df_source.head(8).to_string(index=False))

# GeoJSON menyimpan koordinat dalam urutan [longitude, latitude].
coordinates = aoi_geojson["coordinates"][0]
longitudes = [point[0] for point in coordinates]
latitudes = [point[1] for point in coordinates]
center_lon = sum(longitudes) / len(longitudes)
center_lat = sum(latitudes) / len(latitudes)
print("\nBatas koordinat AOI Bangkalan:")
print(f"Longitude: {min(longitudes):.7f} deg E s/d {max(longitudes):.7f} deg E")
print(f"Latitude : {min(abs(v) for v in latitudes):.7f} deg S s/d {max(abs(v) for v in latitudes):.7f} deg S")
print(f"Titik pusat AOI: {abs(center_lat):.7f} deg LS, {center_lon:.7f} deg BT")





In [ ]:
import os
import numpy as np
import pandas as pd

PROJECT_ROOT = get_project_root()
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
file_path = os.path.join(DATA_DIR, "data_polutan_bangkalan.csv")

required_columns = [
    "Tanggal",
    "NO2 (Harian)",
    "CO (Harian)",
    "SO2 (Harian)",
    "CH4 (Harian)",
]
df_source = pd.read_csv(file_path)
missing_columns = [column for column in required_columns if column not in df_source.columns]
if missing_columns:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing_columns}")

df_raw = df_source[required_columns].copy()
df_raw = df_raw.rename(
    columns={
        "Tanggal": "DATE_TIME",
        "NO2 (Harian)": "NO2",
        "CO (Harian)": "CO",
        "SO2 (Harian)": "SO2",
        "CH4 (Harian)": "CH4",
    }
)
df_raw["DATE_TIME"] = pd.to_datetime(df_raw["DATE_TIME"], errors="coerce")
if df_raw["DATE_TIME"].isna().any():
    raise ValueError("Kolom Tanggal mengandung nilai tanggal yang tidak valid")

df_raw = df_raw.sort_values("DATE_TIME").drop_duplicates("DATE_TIME").reset_index(drop=True)
for column in ["CO", "NO2", "SO2", "CH4"]:
    df_raw[column] = pd.to_numeric(df_raw[column], errors="coerce")

co_raw = df_raw[["DATE_TIME", "CO"]].copy()
print(f"Dataset berhasil dimuat dari: {file_path}")
print(f"Dimensi data: {df_raw.shape[0]} baris x {df_raw.shape[1]} kolom")
print(f"Rentang tanggal: {df_raw['DATE_TIME'].min().date()} s/d {df_raw['DATE_TIME'].max().date()}")
print("Missing values per polutan:")
print(df_raw[["CO", "NO2", "SO2", "CH4"]].isna().sum())
print("\n5 baris pertama data harian:")
print(df_raw.head())


--- 

### 2. Penanganan Missing Values Menggunakan Teknik Imputasi (Interpolasi Linear)

Untuk mempertahankan kontinuitas deret waktu harian 365 hari tanpa membuang baris data tanggal yang hilang, dilakukan **Imputasi Interpolasi Linear** (`interpolate(method='linear')`).

#### Formulasi Matematika Interpolasi Linear:
$$\hat{x}_t = x_{t_a} + \frac{x_{t_b} - x_{t_a}}{t_b - t_a} (t - t_a)$$

#### Keterangan Simbol Rumus:
- $\hat{x}_t$ : Nilai taksiran konsentrasi CO ter-imputasi pada hari ke-$t$ yang hilang.
- $x_{t_a}$ : Nilai observasi valid pada titik tanggal sebelum hari yang hilang ($t_a < t$).
- $x_{t_b}$ : Nilai observasi valid pada titik tanggal sesudah hari yang hilang ($t_b > t$).
- $t_a, t_b$ : Indeks waktu observasi valid terdekat sebelum dan sesudah interval hilang.

#### Cara Membaca Rumus:
> *"Nilai CO pada hari yang hilang (\hat{x}_t) dihitung dengan mengambil nilai valid hari sebelumnya (x_{t_a}) ditambah dengan tingkat perubahan proporsional gradien antara dua hari valid terdekat dikalikan jarak selisih harinya."*

#### Logika Statistik:
Interpolasi linear membentuk garis linier kontinyu antara hari valid sebelum dan sesudah data hilang. Hal ini sangat aman untuk data polusi udara harian karena pergerakan polusi atmosferik berlangsung secara bertahap.



In [ ]:
# Eksekusi Imputasi Interpolasi Linear
co_imputed = co_raw.copy()
co_imputed['CO_imputed'] = co_imputed['CO'].interpolate(method='linear').bfill().ffill()

missing_after = co_imputed['CO_imputed'].isna().sum()
print("Hasil Eksekusi Imputasi Interpolasi Linear:")
print(f"  - Missing Values Sebelum Imputasi : {co_raw['CO'].isna().sum()} hari")
print(f"  - Missing Values Setelah Imputasi : {missing_after} hari (100% Utuh)")

# Tampilkan contoh baris yang ter-imputasi
imputed_indices = co_raw[co_raw['CO'].isna()].index
print("\nContoh 5 Baris Data Hasil Imputasi:")
print(co_imputed.loc[imputed_indices[:5], ['DATE_TIME', 'CO', 'CO_imputed']])


Hasil Eksekusi Imputasi Interpolasi Linear:
  - Missing Values Sebelum Imputasi : 264 hari
  - Missing Values Setelah Imputasi : 0 hari (100% Utuh)

Contoh 5 Baris Data Hasil Imputasi:
   DATE_TIME  CO  CO_imputed
0 2025-08-30 NaN    0.026541
1 2025-08-31 NaN    0.026541
2 2025-09-01 NaN    0.026541
3 2025-09-02 NaN    0.026541
4 2025-09-03 NaN    0.026541


--- 

### 3. Identifikasi dan Penanganan Outlier (Interquartile Range Capping / Winsorization)

Untuk mendeteksi dan membatasi pencilan ekstrem tanpa membuang tanggal observasi, digunakan metode **Interquartile Range (IQR)** yang dilanjutkan dengan **Winsorization Capping**.

#### Formulasi Matematika Metode IQR:
$$\text{IQR} = Q_3 - Q_1$$
$$\text{Batas Bawah (Lower Bound)} = Q_1 - 1{,}5 \times \text{IQR}$$
$$\text{Batas Atas (Upper Bound)} = Q_3 + 1{,}5 \times \text{IQR}$$

#### Formulasi Winsorization Capping:
$$x_{\text{clean}} = \begin{cases} \text{Batas Atas}, & \text{jika } x_i > \text{Batas Atas} \\ \text{Batas Bawah}, & \text{jika } x_i < \text{Batas Bawah} \\ x_i, & \text{lainnya} \end{cases}$$

#### Keterangan Simbol Rumus:
- $Q_1$ : Kuartil Pertama (Persentil ke-25 data CO ter-imputasi).
- $Q_3$ : Kuartil Ketiga (Persentil ke-75 data CO ter-imputasi).
- $\text{IQR}$ : Jangkauan Interkuartil ($Q_3 - Q_1$).
- $1{,}5$ : Pengali standar baku Tukey untuk mendeteksi pencilan moderat.
- $x_{\text{clean}}$ : Nilai konsentrasi CO bersih setelah dibatasi (*capped*).

#### Cara Membaca Rumus:
> *"Batas toleransi outlier dihitung dengan mencari selisih antara Kuartil-3 dan Kuartil-1 (IQR). Setiap nilai CO yang melampaui Batas Atas (Q3 + 1,5 x IQR) dibatasi nilainya menjadi sama dengan Batas Atas tersebut, sehingga tidak merusak distribusi data."*


#### Ringkasan hasil pada data Bangkalan

Setelah imputasi linear, deteksi IQR pada data harian menemukan **0 outlier CO**, **4 outlier NO2**, **13 outlier SO2**, dan **0 outlier CH4**. Nilai outlier dibatasi pada batas IQR tanpa menghapus baris tanggal. Hasil CH4 harus dibaca dengan sangat hati-hati karena hanya terdapat satu nilai harian valid sebelum imputasi.



In [ ]:
import numpy as np
import pandas as pd

# Data CO sudah dimuat dari data_polutan_bangkalan.csv.
co_imputed = co_raw.copy()
co_imputed["CO_imputed"] = (
    co_imputed["CO"].interpolate(method="linear").bfill().ffill()
)

missing_after = int(co_imputed["CO_imputed"].isna().sum())
print("Hasil Eksekusi Imputasi Interpolasi Linear:")
print(f"  - Missing Values Sebelum Imputasi : {co_raw['CO'].isna().sum()} hari")
print(f"  - Missing Values Setelah Imputasi : {missing_after} hari")

# Hitung statistik IQR dan batasi nilai ekstrem tanpa menghapus tanggal.
q1 = co_imputed["CO_imputed"].quantile(0.25)
q3 = co_imputed["CO_imputed"].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

co_clean = co_imputed.copy()
co_clean["pollutant_clean"] = np.clip(
    co_clean["CO_imputed"], lower_bound, upper_bound
)
outliers_detected = co_imputed[
    (co_imputed["CO_imputed"] < lower_bound)
    | (co_imputed["CO_imputed"] > upper_bound)
]

print("Hasil Deteksi dan Perbaikan Outlier (IQR Capping):")
print(f"  - Kuartil 1 (Q1)            : {q1:.6f}")
print(f"  - Kuartil 3 (Q3)            : {q3:.6f}")
print(f"  - Interquartile Range (IQR) : {iqr:.6f}")
print(f"  - Batas Bawah               : {lower_bound:.6f}")
print(f"  - Batas Atas                : {upper_bound:.6f}")
print(f"  - Jumlah Outlier            : {len(outliers_detected)} hari")
print(f"  - Min Pollutant Clean       : {co_clean['pollutant_clean'].min():.6f}")
print(f"  - Max Pollutant Clean       : {co_clean['pollutant_clean'].max():.6f}")


--- 

### 4. Ekstraksi Fitur Deret Waktu Menggunakan Library TSFEL ($f_1$ s/d $f_{68}$)

Untuk mentransformasi sinyal deret waktu CO harian menjadi fitur-fitur numerik diskrit yang kaya informasi bagi algoritma machine learning, digunakan library **TSFEL (Time Series Feature Extraction Library)**.

Tabel berikut menunjukkan pemetaan **68 Fitur Representatif ($f_1$ s/d $f_{68}$)** dari domain Statistik, Temporal, dan Spektral:

| Kode Fitur | Nama Fitur TSFEL | Domain | Deskripsi Fitur |
| :---: | :--- | :---: | :--- |
| **f1** | `Absolute energy` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f2** | `Average power` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f3** | `ECDF Percentile Count_0` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f4** | `ECDF Percentile Count_1` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f5** | `ECDF Percentile_0` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f6** | `ECDF Percentile_1` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f7** | `ECDF_0` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f8** | `ECDF_1` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f9** | `ECDF_2` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f10** | `ECDF_3` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f11** | `ECDF_4` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f12** | `ECDF_5` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f13** | `ECDF_6` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f14** | `ECDF_7` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f15** | `ECDF_8` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f16** | `ECDF_9` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f17** | `Entropy` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f18** | `Histogram mode` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f19** | `Interquartile range` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f20** | `Kurtosis` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f21** | `Max` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f22** | `Mean` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f23** | `Mean absolute deviation` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f24** | `Median` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f25** | `Median absolute deviation` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f26** | `Min` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f27** | `Peak to peak distance` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f28** | `Root mean square` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f29** | `Skewness` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f30** | `Standard deviation` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f31** | `Variance` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f32** | `Area under the curve` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f33** | `Autocorrelation` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f34** | `Centroid` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f35** | `Mean absolute diff` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f36** | `Mean diff` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f37** | `Median absolute diff` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f38** | `Median diff` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f39** | `Negative turning points` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f40** | `Neighbourhood peaks` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f41** | `Positive turning points` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f42** | `Signal distance` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f43** | `Slope` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f44** | `Sum absolute diff` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f45** | `Zero crossing rate` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f46** | `Spectrogram mean coefficient_0.02Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f47** | `Spectrogram mean coefficient_0.03Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f48** | `Spectrogram mean coefficient_0.05Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f49** | `Spectrogram mean coefficient_0.06Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f50** | `Spectrogram mean coefficient_0.08Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f51** | `Spectrogram mean coefficient_0.0Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f52** | `Spectrogram mean coefficient_0.11Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f53** | `Spectrogram mean coefficient_0.13Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f54** | `Spectrogram mean coefficient_0.15Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f55** | `Spectrogram mean coefficient_0.16Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f56** | `Spectrogram mean coefficient_0.18Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f57** | `Spectrogram mean coefficient_0.19Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f58** | `Spectrogram mean coefficient_0.1Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f59** | `Spectrogram mean coefficient_0.21Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f60** | `Spectrogram mean coefficient_0.23Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f61** | `Spectrogram mean coefficient_0.24Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f62** | `Spectrogram mean coefficient_0.26Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f63** | `Spectrogram mean coefficient_0.27Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f64** | `Spectrogram mean coefficient_0.29Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f65** | `Spectrogram mean coefficient_0.31Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f66** | `Spectrogram mean coefficient_0.32Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f67** | `Spectrogram mean coefficient_0.34Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f68** | `Spectrogram mean coefficient_0.35Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |



In [ ]:
import tsfel

# Ekstraksi Fitur TSFEL
cfg_stat = tsfel.get_features_by_domain('statistical')
stat_res = tsfel.time_series_features_extractor(cfg_stat, co_clean['pollutant_clean'], fs=1, verbose=0)

cfg_temp = tsfel.get_features_by_domain('temporal')
temp_res = tsfel.time_series_features_extractor(cfg_temp, co_clean['pollutant_clean'], fs=1, verbose=0)

cfg_spec = tsfel.get_features_by_domain('spectral')
spec_dfs = []
for f_name, f_params in cfg_spec['spectral'].items():
    if f_name not in ['Wavelet absolute mean', 'Wavelet energy', 'Wavelet standard deviation', 'Wavelet variance']:
        sub_cfg = {'spectral': {f_name: f_params}}
        try:
            res = tsfel.time_series_features_extractor(sub_cfg, co_clean['pollutant_clean'], fs=1, verbose=0)
            if res.columns.duplicated().any():
                res.columns = [f"{c}_{i}" for i, c in enumerate(res.columns)]
            spec_dfs.append(res)
        except Exception:
            pass

combined_features = pd.concat([stat_res, temp_res] + spec_dfs, axis=1)

seen = {}
clean_cols = []
for col in combined_features.columns:
    c_name = col.replace('0_', '')
    if c_name in seen:
        seen[c_name] += 1
        clean_cols.append(f"{c_name}_{seen[c_name]}")
    else:
        seen[c_name] = 0
        clean_cols.append(c_name)
combined_features.columns = clean_cols

# Ambil tepat 68 fitur (f1 s/d f68)
features_68 = combined_features.iloc[:, :68].copy()
features_68.columns = [f"f{i+1}" for i in range(68)]

print(f"Ekstraksi Fitur TSFEL Berhasil!")
print(f"Dimensi Matriks Fitur Extracted: {features_68.shape[0]} sampel x {features_68.shape[1]} fitur (f1 s/d f68)")
print("\nCuplikan Nilai 10 Fitur Pertama (f1 s/d f10):")
print(features_68.iloc[:, :10])


Ekstraksi Fitur TSFEL Berhasil!
Dimensi Matriks Fitur Extracted: 1 sampel x 68 fitur (f1 s/d f68)

Cuplikan Nilai 10 Fitur Pertama (f1 s/d f10):
         f1        f2    f3     f4        f5        f6       f7        f8  \
0  0.325726  0.000895  73.0  292.0  0.026157  0.032683  0.00274  0.005479   

         f9       f10  
0  0.008219  0.010959  


--- 

### 5. Analisis Kemiripan Sinyal Polutan (Similarity Analysis)

Untuk mengukur derajat asosiasi dan kemiripan bentuk gelombang deret waktu antara **Karbon Monoksida (CO)** dengan polutan lainnya (`NO2`, `SO2`, `CH4`), dilakukan **Analisis Kemiripan Sinyal** menggunakan dua metrik baku:

#### 1. Koefisien Korelasi Pearson ($r$):
$$r_{XY} = \frac{\sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i=1}^{n} (x_i - \bar{x})^2} \cdot \sqrt{\sum_{i=1}^{n} (y_i - \bar{y})^2}}$$

#### 2. Cosine Similarity ($S_C$):
$$S_C(\mathbf{A}, \mathbf{B}) = \frac{\mathbf{A} \cdot \mathbf{B}}{\|\mathbf{A}\| \|\mathbf{B}\|} = \frac{\sum_{i=1}^{n} A_i B_i}{\sqrt{\sum_{i=1}^{n} A_i^2} \cdot \sqrt{\sum_{i=1}^{n} B_i^2}}$$

#### Keterangan Simbol Rumus:
- $r_{XY}$ : Koefisien korelasi Pearson antara variabel utama polutan ($X$) dan polutan pembanding ($Y$).
- $S_C$ : Nilai kemiripan sudut kosinus antara vektor sinyal polutan dan polutan pembanding ($-1 \le S_C \le 1$).
- $\mathbf{A}, \mathbf{B}$ : Vektor deret waktu konsentrasi polutan harian.

#### Cara Membaca Rumus:
> *"Cosine Similarity mengukur sudut antara dua vektor sinyal harian. Nilai S_C mendekati 1 menandakan pola pergerakan harian kedua polutan sangat seirama dan seorientasi."*


#### Hasil pada dataset Bangkalan

| Parameter | Korelasi Pearson terhadap CO | Cosine Similarity terhadap CO |
| :--- | ---: | ---: |
| CO | 1,000000 | 1,000000 |
| NO2 | -0,021329 | 0,668772 |
| SO2 | 0,027981 | -0,138142 |
| CH4 | Tidak terdefinisi | 0,992183 |

Korelasi Pearson CH4 tidak terdefinisi karena deret CH4 hasil imputasi konstan (hanya satu observasi harian valid). Nilai Cosine Similarity CH4 juga tidak menunjukkan pola harian yang andal, sehingga perlu ditafsirkan bersama keterbatasan tersebut.




In [ ]:
import numpy as np
from numpy.linalg import norm
import pandas as pd

# Gunakan data_polutan_bangkalan.csv yang sudah dimuat pada df_raw.
# Semua polutan disejajarkan berdasarkan tanggal yang sama.
comparison_columns = ["CO", "NO2", "SO2", "CH4"]
df_all = df_raw[["DATE_TIME"] + comparison_columns].copy()
for column in comparison_columns:
    df_all[column] = (
        pd.to_numeric(df_all[column], errors="coerce")
        .interpolate(method="linear")
        .bfill()
        .ffill()
    )

pearson_corr = df_all[comparison_columns].corr()["CO"]


def cosine_sim(a, b):
    denominator = norm(a) * norm(b)
    return float(np.dot(a, b) / denominator) if denominator else np.nan


cosine_sims = {}
co_vec = df_all["CO"].to_numpy(dtype=float)
for column in comparison_columns:
    cosine_sims[column] = cosine_sim(co_vec, df_all[column].to_numpy(dtype=float))

similarity_df = pd.DataFrame(
    {
        "Parameter Polutan": comparison_columns,
        "Pearson Correlation (r)": [pearson_corr[column] for column in comparison_columns],
        "Cosine Similarity (Sc)": [cosine_sims[column] for column in comparison_columns],
    }
)

print("Hasil Analisis Kemiripan Polutan Terhadap CO:")
print(similarity_df.to_string(index=False))


--- 

### 6. Penyimpanan dan Integrasi Dataset Terpreparasi

Dataset terpreparasi dan fitur TSFEL disimpan dalam format CSV di folder `data`:
1. **`data_prepared_<polutan>.csv`**: Data harian per polutan setelah imputasi dan penanganan IQR pada alur ekstraksi akhir.
2. **`features_tsfel_<polutan>.csv`**: Fitur TSFEL hasil ekstraksi untuk masing-masing polutan.

File utama `data_prepared.csv` dan `features_tsfel.csv` disediakan sebagai keluaran utama untuk CO. Nilai hasil imputasi merupakan taksiran; khususnya parameter dengan proporsi nilai kosong tinggi perlu ditafsirkan hati-hati.


In [ ]:
# Penyimpanan dataset terpreparasi ke folder data utama
p1 = os.path.join(DATA_DIR, 'data_prepared.csv')
p2 = os.path.join(DATA_DIR, 'features_tsfel.csv')

co_clean[['DATE_TIME', 'pollutant_clean']].to_csv(p1, index=False)
features_68.to_csv(p2, index=False)

print(f"Berhasil menyimpan file ke folder data utama ({DATA_DIR}):")
print(f"  - File Prepared Data : '{p1}'")
print(f"  - File TSFEL Features: '{p2}'")

print("\nStatus Data Preparation: 100% SELESAI & SIAP UNTUK TAHAP MODELING!")

Berhasil menyimpan file ke folder data utama (d:\PSD-B-main\PSD-B-main\data):
  - File Prepared Data : 'd:\PSD-B-main\PSD-B-main\data\data_prepared.csv'
  - File TSFEL Features: 'd:\PSD-B-main\PSD-B-main\data\features_tsfel.csv'

Status Data Preparation: 100% SELESAI & SIAP UNTUK TAHAP MODELING!


In [ ]:
import os
import numpy as np
import pandas as pd
import tsfel

# Gunakan dataset CSV Bangkalan yang sama dengan sel sebelumnya.
PROJECT_ROOT = get_project_root()
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
file_path = os.path.join(DATA_DIR, "data_polutan_bangkalan.csv")

source = pd.read_csv(file_path)
column_map = {
    "Tanggal": "DATE_TIME",
    "NO2 (Harian)": "NO2",
    "CO (Harian)": "CO",
    "SO2 (Harian)": "SO2",
    "CH4 (Harian)": "CH4",
}
missing_columns = [column for column in column_map if column not in source.columns]
if missing_columns:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing_columns}")

df_raw = source[list(column_map)].rename(columns=column_map)
df_raw["DATE_TIME"] = pd.to_datetime(df_raw["DATE_TIME"], errors="coerce")
if df_raw["DATE_TIME"].isna().any():
    raise ValueError("Kolom Tanggal mengandung nilai yang tidak valid")

pollutants = ["CO", "SO2", "CH4", "NO2"]
for pol in pollutants:
    df_raw[pol] = pd.to_numeric(df_raw[pol], errors="coerce")
df_raw = (
    df_raw.sort_values("DATE_TIME")
    .drop_duplicates("DATE_TIME")
    .reset_index(drop=True)
)

# Ekstraksi per domain mencegah konflik nama fitur pada TSFEL versi terbaru.
cfg_stat = tsfel.get_features_by_domain("statistical")
cfg_temp = tsfel.get_features_by_domain("temporal")
cfg_spec = tsfel.get_features_by_domain("spectral")


def extract_tsfel_features(signal):
    results = [
        tsfel.time_series_features_extractor(cfg_stat, signal, fs=1, verbose=0),
        tsfel.time_series_features_extractor(cfg_temp, signal, fs=1, verbose=0),
    ]
    for feature_name, feature_params in cfg_spec["spectral"].items():
        if feature_name in {
            "Wavelet absolute mean",
            "Wavelet energy",
            "Wavelet standard deviation",
            "Wavelet variance",
        }:
            continue
        try:
            result = tsfel.time_series_features_extractor(
                {"spectral": {feature_name: feature_params}},
                signal,
                fs=1,
                verbose=0,
            )
            results.append(result)
        except Exception:
            continue

    features = pd.concat(results, axis=1)
    unique_columns = []
    seen = {}
    for column in features.columns:
        base_name = column.replace("0_", "")
        count = seen.get(base_name, 0)
        unique_columns.append(base_name if count == 0 else f"{base_name}_{count}")
        seen[base_name] = count + 1
    features.columns = unique_columns

    # Kurtosis dan skewness tidak terdefinisi untuk sinyal konstan seperti CH4.
    # Ubah NaN/inf pada fitur turunan menjadi angka netral agar CSV siap dipakai.
    features = features.apply(pd.to_numeric, errors="coerce")
    features = features.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return features


for pol in pollutants:
    print(f"\n================ MENGOLAH POLUTAN: {pol} ================")
    values = df_raw[pol].interpolate(method="linear").bfill().ffill()
    if values.isna().all():
        print(f"[WARNING] {pol} tidak memiliki nilai valid. Dilewati...")
        continue

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    values = values.clip(lower=lower_bound, upper=upper_bound)

    df_pol = pd.DataFrame(
        {"DATE_TIME": df_raw["DATE_TIME"], "pollutant_clean": values}
    )
    p1 = os.path.join(DATA_DIR, f"data_prepared_{pol.lower()}.csv")
    p2 = os.path.join(DATA_DIR, f"features_tsfel_{pol.lower()}.csv")
    df_pol.to_csv(p1, index=False)

    print(f"IQR capping: batas bawah={lower_bound:.6f}, batas atas={upper_bound:.6f}")
    print(f"Sedang mengekstrak fitur TSFEL untuk {pol}...")
    features_pol = extract_tsfel_features(values.to_numpy(dtype=float))
    features_pol.to_csv(p2, index=False)
    non_finite_count = int(~np.isfinite(features_pol.to_numpy()).all())

    print(f"Berhasil menyimpan file untuk {pol}:")
    print(f"  - Prepared Data : {os.path.basename(p1)} ({len(df_pol)} baris)")
    print(f"  - TSFEL Features: {os.path.basename(p2)} ({features_pol.shape[1]} fitur)")
    print(f"  - Nilai NaN/inf pada fitur: {non_finite_count}")

print("\nStatus Data Preparation semua polutan: SELESAI")





In [ ]:
# Eksekusi imputasi polynomial orde 2 pada data harian

import os
import numpy as np
import pandas as pd

polynomial_order = 2
pollutants = ["CO", "NO2", "SO2", "CH4"]

# Urutkan tanggal dan pastikan satu baris per tanggal.
df_polynomial = (
    df_raw.sort_values("DATE_TIME")
    .drop_duplicates("DATE_TIME")
    .reset_index(drop=True)
    .copy()
)

def polynomial_impute(series, dates, order=2):
    values = pd.to_numeric(series, errors="coerce").replace(
        [np.inf, -np.inf], np.nan
    )

    if values.notna().sum() < order + 1:
        raise ValueError(
            f"Imputasi polynomial orde {order} membutuhkan minimal "
            f"{order + 1} nilai valid; hanya ditemukan {values.notna().sum()}."
        )

    # Gunakan jarak hari sebagai sumbu interpolasi.
    day_index = pd.DatetimeIndex(dates).map(pd.Timestamp.toordinal).to_numpy()
    dated_values = pd.Series(values.to_numpy(), index=day_index)

    imputed = dated_values.interpolate(
        method="polynomial",
        order=order,
        limit_area="inside",
    )

    # Polynomial tidak mengisi bagian awal/akhir deret; gunakan nilai terdekat
    # untuk missing value pada bagian tersebut.
    imputed = imputed.bfill().ffill()

    return pd.Series(imputed.to_numpy(), index=series.index)


for column in pollutants:
    missing_before = int(
        pd.to_numeric(df_polynomial[column], errors="coerce").isna().sum()
    )
    df_polynomial[column] = polynomial_impute(
        df_polynomial[column],
        df_polynomial["DATE_TIME"],
        order=polynomial_order,
    )
    missing_after = int(df_polynomial[column].isna().sum())

    print(
        f"{column}: missing sebelum={missing_before}, "
        f"sesudah={missing_after}"
    )

output_path = os.path.join(DATA_DIR, "data_prepared_polynomial.csv")
df_polynomial.to_csv(output_path, index=False)

print(f"\nHasil imputasi polynomial tersimpan di: {output_path}")